# Queens 8 × 8 · Adaptive sampling with multiplicative risk

Load the trained checkpoint from `01_queens_discrete_flow_matching.ipynb`.
At each iteration, order cells by their predicted EMPTY probabilities and choose
the longest prefix satisfying the product-risk budget. Generation ends with
eight colored cells.

Inspect trajectories, estimate success across repeated attempts and measure
the model calls used by each trajectory.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").exists()
)
sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))

import ipywidgets as widgets
import numpy as np
import plotly.graph_objects as go
import torch
from IPython.display import HTML, display
from plotly.subplots import make_subplots
from _shared.queens_data import BOARD_SIZE, EMPTY, load_queens_dataset, solution_checks
from _shared.queens_confidence_sampling import (
    load_trained_dfm,
    retry_statistics,
)
from _shared.queens_adaptive_sampling import sample_product_risk_dfm
from _shared.queens_sampling_plots import risk_retry_plot

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ARTIFACTS = PROJECT_ROOT / "artifacts" / "queens8"
CHECKPOINT_PATH = ARTIFACTS / "queens8_dfm.pt"
RESULT_PATH = ARTIFACTS / "queens8_dfm_product_risk_evaluation.npz"
PLOT_CONFIG = dict(responsive=True, displaylogo=False, displayModeBar=False)
NUM_CLASSES = BOARD_SIZE + 1
PALETTE = [
    "#B8A5E8",
    "#F0B48B",
    "#92C9E8",
    "#A9D4A4",
    "#E8CD83",
    "#ECA6BB",
    "#9FB9D4",
    "#C7B8A3",
    "#F1F3F5",
]
CATEGORICAL_SCALE = [
    [position, color]
    for index, color in enumerate(PALETTE)
    for position in (index / NUM_CLASSES, (index + 1) / NUM_CLASSES)
]

## 1. Load the trained model

Load the prepared dataset, trained weights and saved test split. The loader
checks the dataset fingerprint. The saved architecture determines whether the model uses scalar time.

The model outputs one erasure logit per cell; sigmoid gives the probability
of EMPTY in the endpoint.


In [ ]:
puzzles = load_queens_dataset(PROJECT_ROOT / "data" / "queens8")
regions = puzzles["regions"]
model, splits, model_info = load_trained_dfm(CHECKPOINT_PATH, puzzles, device=DEVICE)
SEED = model_info["seed"]
RISK_BUDGETS = (0.01, 0.025, 0.05, 0.1, 0.25, 0.5, 1.0)
display(
    HTML(
        f"<b>Loaded trained DFM</b> · {DEVICE} · {model_info['optimizer_updates']:,} updates<br>"
        f"Time conditioning: {model_info['use_time_conditioning']}<br>"
        f"{len(regions):,} puzzles · {len(splits['test']):,} held-out puzzles"
    )
)

In [3]:
def add_board(fig, tokens, board, row, col, queens=False):
    tokens, board = np.asarray(tokens), np.asarray(board)
    fig.add_trace(
        go.Heatmap(
            z=tokens,
            zmin=-0.5,
            zmax=NUM_CLASSES - 0.5,
            colorscale=CATEGORICAL_SCALE,
            showscale=False,
            xgap=1,
            ygap=1,
            customdata=board,
            hovertemplate="row=%{y}, col=%{x}<br>region=%{customdata}<br>token=%{z}<extra></extra>",
        ),
        row=row,
        col=col,
    )
    xs, ys = [], []
    for r in range(BOARD_SIZE):
        for c in range(BOARD_SIZE):
            if c + 1 < BOARD_SIZE and board[r, c] != board[r, c + 1]:
                xs.extend([c + 0.5, c + 0.5, None])
                ys.extend([r - 0.5, r + 0.5, None])
            if r + 1 < BOARD_SIZE and board[r, c] != board[r + 1, c]:
                xs.extend([c - 0.5, c + 0.5, None])
                ys.extend([r + 0.5, r + 0.5, None])
    fig.add_trace(
        go.Scatter(
            x=xs,
            y=ys,
            mode="lines",
            line=dict(color="#64748B", width=1.5),
            hoverinfo="skip",
            showlegend=False,
        ),
        row=row,
        col=col,
    )
    if queens:
        qr, qc = np.where(tokens != EMPTY)
        fig.add_trace(
            go.Scatter(
                x=qc,
                y=qr,
                mode="text",
                text=["♛"] * len(qr),
                textfont=dict(size=25, color="#172033"),
                hoverinfo="skip",
                showlegend=False,
            ),
            row=row,
            col=col,
        )
    fig.update_xaxes(
        range=[-0.5, BOARD_SIZE - 0.5],
        showgrid=False,
        zeroline=False,
        showticklabels=False,
        constrain="domain",
        fixedrange=True,
        row=row,
        col=col,
    )
    xaxis_name = fig.get_subplot(row, col).xaxis.plotly_name.replace("axis", "")
    fig.update_yaxes(
        range=[BOARD_SIZE - 0.5, -0.5],
        showgrid=False,
        zeroline=False,
        showticklabels=False,
        scaleanchor=xaxis_name,
        scaleratio=1,
        constrain="domain",
        fixedrange=True,
        row=row,
        col=col,
    )


def style_boards(fig, title, rows=1):
    fig.update_layout(
        title=dict(text=title, x=0, font=dict(size=22)),
        template="plotly_white",
        autosize=True,
        height=rows * 225 + 90,
        margin=dict(l=10, r=10, t=85, b=20),
        font=dict(family="Arial, sans-serif", size=15),
        showlegend=False,
    )
    fig.update_annotations(font=dict(size=14))
    return fig

## 2. Adaptive deletion rule

Let $M$ be the number of remaining colored cells. Time follows the fraction of
required deletions already completed:

$$
t=\frac{64-M}{56}.
$$

One model call predicts an endpoint EMPTY probability $p_i=\sigma(\ell_i)$ for
each colored cell. Keep these predictions fixed while choosing the deletion set.

### Sequential selection with a budget

Associate each cell with a nonnegative cost and convert the risk budget into a
cost budget:

$$
c_i=-\log p_i=\operatorname{softplus}(-\ell_i),
\qquad B=-\log(1-\delta).
$$

Start with an empty set $S$ and total cost $C=0$:

1. Draw one cell from the pool $R$ of unselected colored cells, with probability
   $P(i\text{ next}\mid R)=p_i/\sum_{j\in R}p_j$.
2. If $C+c_i\leq B$, add the cell to $S$, remove it from $R$, and accumulate
   its cost: $C\leftarrow C+c_i$.
3. Repeat until the next cell would exceed the budget or $M-8$ cells have been
   selected. Stop at the first budget failure; do not skip that cell and search
   for a cheaper one.

The accumulated cost corresponds to the factorized risk

$$
1-\prod_{i\in S}p_i
=1-\exp\left(-\sum_{i\in S}c_i\right).
$$

Under a conditional-independence approximation, this estimates the probability
that at least one selected cell should contain a queen. It is a local risk
estimate, not a guarantee of a valid final solution.

### Equivalent Gumbel order and prefix

The same distribution of weighted draws without replacement can be generated
by one random sort. Independently for every colored cell, sample

$$
u_i\sim U(0,1),\qquad
g_i=-\log(-\log u_i),\qquad
s_i=\log p_i+g_i.
$$

Sort $s_i$ in descending order to obtain $\pi$. This is the
[Gumbel-Top-k trick](https://proceedings.mlr.press/v97/kool19a.html): the first
cell is selected proportionally to its weight $p_i$, the second proportionally
to the remaining weights, and so on. Thus the sorted order has the same
distribution as the sequential selection above.

Compute the cumulative costs along that order and choose its longest admissible
prefix:

$$
C_k=\sum_{j=1}^{k}c_{\pi_j},
\qquad
m=\max\left\{k\in\{0,\ldots,M-8\}:C_k\leq B\right\}.
$$

The implementation batches score generation, sorting and cumulative sums across
trajectories instead of drawing cells one by one. Since costs are nonnegative,
the prefix ends at exactly the first budget failure of the sequential procedure.
The budget determines the prefix length; a fixed deletion count is not required.

Compute $\log p_i$ directly with `logsigmoid` and compare cumulative costs to
`-log1p(-delta)`. Only trajectory recording reconstructs the risk, using
`-expm1(-C)` to remain accurate near zero.

The budget must satisfy $0\leq\delta\leq1$. At $\delta=1$, $B=+\infty$ and the
prefix is limited only by the remaining deletion count. If $m=0$, erase the
first sampled cell and record a **forced step**, which may exceed the budget.

Apply the selected deletions together, then call the model again for the new
state. Each iteration deletes at least one cell and at most $M-8$, so generation
finishes with exactly eight cells in 1–56 model calls. Each trajectory has its
own time and deletion count; finished trajectories leave the active batch.
Evaluation checks rows, columns, regions and the non-touching rule.


## 3. Solution trajectories

Rows show three risk budgets for the same held-out puzzle. Panels are sampled
from the iteration history, including the initial and final states. Labels show
the model-call index, time and number of deletions. A star marks a forced step.
The final panel reports validity and total model calls.


In [4]:
N_TRAJECTORY_PUZZLES = 2
TRAJECTORY_RISK_BUDGETS = (0.01, 0.05, 0.25)
N_TRAJECTORY_PANELS = 6
for number, index in enumerate(splits["test"][:N_TRAJECTORY_PUZZLES]):
    board = regions[int(index)]
    runs = []
    for risk_budget in TRAJECTORY_RISK_BUDGETS:
        result = sample_product_risk_dfm(
            model,
            board,
            risk_budget=risk_budget,
            seed=SEED + 600 + number,
            record_trajectory=True,
        )
        history = result["trajectory"]
        positions = np.rint(
            np.linspace(0, len(history) - 1, min(N_TRAJECTORY_PANELS, len(history)))
        ).astype(int)
        frames = [history[position] for position in positions]
        solved = bool(solution_checks(board, result["tokens"][0].numpy())["solved"])
        runs.append((risk_budget, result, frames, solved))
    panel_count = max(len(frames) for _, _, frames, _ in runs)
    titles = []
    for risk_budget, result, frames, solved in runs:
        for panel in range(panel_count):
            if panel >= len(frames):
                titles.append("")
                continue
            frame = frames[panel]
            if panel == 0:
                title = f"δ = {risk_budget:g} · initial"
            elif panel == len(frames) - 1:
                title = (
                    "Solved" if solved else "Not solved"
                ) + f" · {int(result['iterations'][0])} calls"
            else:
                star = " *" if frame["forced"] else ""
                title = f"k = {frame['iteration']} · t = {frame['time']:.2f}<br>−{frame['deleted']} cells{star}"
            titles.append(title)
    fig = make_subplots(
        rows=len(TRAJECTORY_RISK_BUDGETS),
        cols=panel_count,
        subplot_titles=titles,
        horizontal_spacing=0.025,
        vertical_spacing=0.12,
    )
    for row, (risk_budget, result, frames, solved) in enumerate(runs, 1):
        for col, frame in enumerate(frames, 1):
            add_board(
                fig, frame["state"].numpy(), board, row, col, queens=col == len(frames)
            )
        for col in range(len(frames) + 1, panel_count + 1):
            fig.update_xaxes(visible=False, row=row, col=col)
            fig.update_yaxes(visible=False, row=row, col=col)
    style_boards(
        fig,
        f"Multiplicative risk · {puzzles['ids'][int(index)]}",
        rows=len(TRAJECTORY_RISK_BUDGETS),
    ).show(config=PLOT_CONFIG)

## 4. Success probability and sampling cost

Evaluate seven risk budgets on the same 256 held-out puzzles, with 32 stochastic
attempts per puzzle and budget. Shared seeds pair randomness across budgets.
Every attempt starts from a complete coloring.

For $c$ successes among $A$ attempts, average over subsets of $k$ attempts:

$$
\widehat{\mathrm{pass@}k}=1-\frac{\binom{A-c}{k}}{\binom{A}{k}},
\qquad 0\leq k\leq A.
$$

Average across puzzles and bootstrap whole puzzles for 95% intervals. The same
resampling weights are used for all budgets. Record validity, model-call counts
and forced steps per attempt. The progress bar counts completed cell deletions.


In [ ]:
ATTEMPTS_PER_PUZZLE = 32
EVAL_BATCH_SIZE = 128
EVAL_SEED = SEED + 10_000
if ATTEMPTS_PER_PUZZLE < 1 or EVAL_BATCH_SIZE < 1:
    raise ValueError("Attempt count and batch size must be positive.")
if (
    not RISK_BUDGETS
    or len(set(RISK_BUDGETS)) != len(RISK_BUDGETS)
    or any(not np.isfinite(b) or not 0 <= b <= 1 for b in RISK_BUDGETS)
):
    raise ValueError(
        "Risk budgets must be distinct finite values between zero and one."
    )
evaluation_indices = splits["test"].copy()
flat_indices = np.repeat(evaluation_indices, ATTEMPTS_PER_PUZZLE)
shape = (len(RISK_BUDGETS), len(evaluation_indices), ATTEMPTS_PER_PUZZLE)
successes = np.empty(shape, dtype=bool)
model_calls = np.empty(shape, dtype=np.int64)
forced_steps = np.empty(shape, dtype=np.int64)
evaluation_complete = False
deletions_per_attempt = BOARD_SIZE**2 - BOARD_SIZE
budget_work = len(flat_indices) * deletions_per_attempt
evaluation_progress = widgets.IntProgress(
    value=0,
    max=budget_work * len(RISK_BUDGETS),
    description="Evaluation",
    layout=widgets.Layout(width="100%"),
)
evaluation_status = widgets.HTML()
display(evaluation_progress, evaluation_status)
completed_work = 0
for budget_index, risk_budget in enumerate(RISK_BUDGETS):
    outcome_batches, call_batches, forced_batches = [], [], []
    for start in range(0, len(flat_indices), EVAL_BATCH_SIZE):
        indices = flat_indices[start : start + EVAL_BATCH_SIZE]
        evaluation_status.value = (
            f"δ = {risk_budget:g} · budget {budget_index + 1}/{len(RISK_BUDGETS)} · "
            f"{start:,}/{len(flat_indices):,} trajectories"
        )

        def update_evaluation(done, total, start=start, offset=completed_work):
            evaluation_progress.value = offset + start * deletions_per_attempt + done

        result = sample_product_risk_dfm(
            model,
            regions[indices],
            risk_budget=risk_budget,
            seed=EVAL_SEED + start,
            progress=update_evaluation,
        )
        outcome_batches.append(
            solution_checks(regions[indices], result["tokens"].numpy())["solved"]
        )
        call_batches.append(result["iterations"].numpy())
        forced_batches.append(result["forced_steps"].numpy())
    target_shape = (len(evaluation_indices), ATTEMPTS_PER_PUZZLE)
    successes[budget_index] = np.concatenate(outcome_batches).reshape(target_shape)
    model_calls[budget_index] = np.concatenate(call_batches).reshape(target_shape)
    forced_steps[budget_index] = np.concatenate(forced_batches).reshape(target_shape)
    completed_work += budget_work
    evaluation_progress.value = completed_work
evaluation_complete = True
evaluation_progress.bar_style = "success"
evaluation_status.value = (
    f"Completed {len(RISK_BUDGETS)} budgets · {len(evaluation_indices):,} puzzles · "
    f"{ATTEMPTS_PER_PUZZLE} attempts each · {model_calls.sum():,} total model calls"
)
evaluation_metadata = dict(
    risk_budgets=np.asarray(RISK_BUDGETS),
    dataset_indices=evaluation_indices,
    puzzle_ids=puzzles["ids"][evaluation_indices],
    evaluation_seed=EVAL_SEED,
    batch_size=EVAL_BATCH_SIZE,
    checkpoint_sha256=model_info["checkpoint_sha256"],
    dataset_sha256=puzzles["metadata"]["data_sha256"],
    optimizer_updates=model_info["optimizer_updates"],
    use_time_conditioning=model_info["use_time_conditioning"],
    sampler_version="queens8_adaptive_product_risk_v1",
    total_erasures=deletions_per_attempt,
)
np.savez_compressed(
    RESULT_PATH,
    successes=successes,
    model_calls=model_calls,
    forced_steps=forced_steps,
    **evaluation_metadata,
)

### Success across attempts

Color identifies the risk budget. Select budgets independently in the legend
panel and enable the 95% intervals in Layers. Selection updates visibility while
retaining the recorded results.


In [6]:
if not evaluation_complete:
    raise RuntimeError("Finish the evaluation cell before plotting.")
statistics = retry_statistics(successes, bootstrap_samples=3000, seed=EVAL_SEED + 1)
np.savez_compressed(
    RESULT_PATH,
    successes=successes,
    model_calls=model_calls,
    forced_steps=forced_steps,
    **evaluation_metadata,
    attempt_counts=statistics["attempts"],
    pass_at_k=statistics["mean"],
    lower=statistics["lower"],
    upper=statistics["upper"],
)
product_risk_view = risk_retry_plot(
    statistics,
    RISK_BUDGETS,
    previous=globals().get("product_risk_view"),
)
display(product_risk_view)

SamplingComparison(children=(FigureWidget({
    'data': [{'fillcolor': 'rgba(117,167,255,0.08)',
             …

### Model calls per attempt

Mean model calls measure sampling cost. Error bars show the 5th–95th percentiles
across trajectories. Hover also reports the fraction of calls that forced a
single deletion because the sampled prefix exceeded the risk budget.


In [7]:
means = model_calls.mean(axis=(-2, -1))
low, high = np.quantile(model_calls, [0.05, 0.95], axis=(-2, -1))
forced_fraction = forced_steps.sum(axis=(-2, -1)) / model_calls.sum(axis=(-2, -1))
cost_fig = go.Figure(
    go.Scatter(
        x=RISK_BUDGETS,
        y=means,
        mode="lines+markers",
        line=dict(color="#75A7FF", width=2.5),
        error_y=dict(
            type="data",
            symmetric=False,
            array=np.maximum(high - means, 0),
            arrayminus=np.maximum(means - low, 0),
        ),
        customdata=np.column_stack((low, high, 100 * forced_fraction)),
        hovertemplate=(
            "δ = %{x:g}<br>Mean calls: %{y:.2f}<br>5–95%: [%{customdata[0]:.0f}, %{customdata[1]:.0f}]"
            "<br>Forced calls: %{customdata[2]:.1f}%<extra></extra>"
        ),
    )
)
cost_fig.update_layout(
    template="plotly_dark",
    paper_bgcolor="#1f1f1f",
    plot_bgcolor="#1f1f1f",
    title="Model calls per attempt",
    height=460,
    autosize=True,
    showlegend=False,
    xaxis=dict(
        title="Risk budget δ", type="log" if min(RISK_BUDGETS) > 0 else "linear"
    ),
    yaxis=dict(title="Model calls", range=[0, deletions_per_attempt + 1]),
    font=dict(family="Arial, sans-serif", size=16),
    margin=dict(l=75, r=25, t=65, b=75),
)
cost_fig.show(config=PLOT_CONFIG)